In [ ]:
%cd ..

import sys
import os

import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import (
    Trainer,
    TrainingArguments,
    AutoConfig, 
    AutoModelForCausalLM, 
    AutoModelForImageTextToText, 
    AutoProcessor, 
    BitsAndBytesConfig
)
from nuscenes.nuscenes import NuScenes
from data.nuscenes_data import NuscenesData
from utils.caption_utils import reason_generate

# MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"  # You can change this ID anytime for testing
MODEL_ID = "/home/ximeng/Documents/NaviDrive/checkpoints/qwen_vl_2B_sft"  # Path to your SFT checkpoint
BASE_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
DATA_PATH = "/home/ximeng/Dataset/nuscenes_full_v1_0/"
VERSION = "v1.0-trainval"  # Recommend using 'v1.0-mini' for initial testing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
nusc = NuScenes(version=VERSION, dataroot=DATA_PATH)
dataset = NuscenesData(nusc, is_train=1, pre_frames=4, future_frames=12)

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert autonomous driving navigator. Your task is to analyze a 360-degree surround-view driving environment and provide concise, safety-oriented driving guidance.\n"
    "Guidelines:\n"
    "1. Coordinate System: The x-axis positive is forward, the y-axis positive is left.\n"
    "2. Attention Priority: Focus on 'Dynamic Hazards' (pedestrians, moving vehicles) and 'Traffic Regulators' (lights, signs, lane markings) on the front cameras.\n"
    "3. Output Format: Start with a concise 'Perception' summary, followed by 'Action', and a brief 'Reasoning'. Do not use '*' in the response."
)

print(f"Loading model: {MODEL_ID}...")
model_config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, 
    dtype=torch.bfloat16, 
    device_map=DEVICE,
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=128*28*28, max_pixels=224*28*28)

In [ ]:
def test_inference_by_index(idx):
    sample = dataset[idx]
    token = sample['token']
    raw_images = sample['raw_images'][-1] # Get 6 images for the current timestamp
    
    # Arrange in surround-view order: [FL, F, FR, BR, B, BL]
    new_order = [2, 3, 4, 5, 0, 1]
    pil_images = []
    for i in new_order:
        img_np = raw_images[i].permute(1, 2, 0).cpu().numpy().astype('uint8')
        pil_images.append(Image.fromarray(img_np))
    
    # Concatenate State Prompt
    wp_past = ", ".join([f"({pt[0]:.2f}, {pt[1]:.2f}, {pt[2]:.2f})" for pt in sample['pre_waypoints']])
    ego_status_prompt = (
        "Current Dynamics:\n"
        f"- Velocity: {sample['velocity'][-1].item():.2f} m/s.\n"
        f"- Yaw Rate: {sample['yaw_rate'][-1].item():.2f} rad/s.\n"
        f"- Acceleration (Longitudinal x, Lateral y): ({sample['accel'][-1][0]:.2f}, {sample['accel'][-1][1]:.2f}) m/s^2.\n"
        f"- Past Trajectory (2Hz): {wp_past} m.\n\n"
    )
    
    user_prompt = (
        "Inputs: 6 images (Full Surround View) and Ego-Vehicle Status.\n"
        "1:FRONT_LEFT, 2:FRONT, 3:FRONT_RIGHT, 4:BACK_RIGHT, 5:BACK, 6:BACK_LEFT.\n"
        f"{ego_status_prompt}"
        "Task: Analyze the current situation and provide the safest next action with reasons."
    )

    print(f"--- [Index: {idx} | Token: {token}] ---")
    
    # Call generation
    _, response = reason_generate(
        user=user_prompt,
        system=SYSTEM_PROMPT,
        images=pil_images,
        processor=processor,
        model=model,
        do_sample=True,
        temperature=0.7 # Lower randomness for testing
    )
    
    # Stitch and display 6 images for manual comparison
    combined_img = Image.new('RGB', (256*6, 256))
    for i, im in enumerate(pil_images):
        combined_img.paste(im.resize((256, 256)), (i*256, 0))
    
    display(combined_img) # Display image in Notebook
    print(f"\nModel Response:\n{response}")

In [ ]:
test_inference_by_index(323)